In [8]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

# ratings.dat 로드 (콜론 2개로 구분)
ratings = pd.read_csv('../data/raw/ratings.dat', sep='::',
                      names=['userId','movieId','rating','timestamp'],
                      engine='python')

# movies.dat 로드
movies = pd.read_csv('../data/raw/movies.dat', sep='::',
                     names=['movieId','title','genres'],
                     engine='python', encoding='latin-1')

print(f"평점 수:  {len(ratings):,}")
print(f"유저 수:  {ratings['userId'].nunique():,}")
print(f"영화 수:  {ratings['movieId'].nunique():,}")
print(f"평점 범위: {ratings['rating'].min()} ~ {ratings['rating'].max()}")
print(f"\n기간: {pd.to_datetime(ratings['timestamp'], unit='s').min().date()} ~ "
      f"{pd.to_datetime(ratings['timestamp'], unit='s').max().date()}")
print(ratings.head())

평점 수:  1,000,209
유저 수:  6,040
영화 수:  3,706
평점 범위: 1 ~ 5

기간: 2000-04-25 ~ 2003-02-28
   userId  movieId  rating  timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291


In [9]:
# 타임스탬프 변환
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')

# 중복 제거
before = len(ratings)
ratings = ratings.sort_values('timestamp').drop_duplicates(
    subset=['userId','movieId'], keep='last'
)
print(f"중복 제거: {before - len(ratings):,}개 제거됨")
print(f"정제 후 평점 수: {len(ratings):,}")

# 시간 기준 80:20 분리
ratings_sorted = ratings.sort_values('timestamp').reset_index(drop=True)
split_idx = int(len(ratings_sorted) * 0.8)
train_df = ratings_sorted.iloc[:split_idx].copy()
test_df  = ratings_sorted.iloc[split_idx:].copy()

assert test_df['timestamp'].min() >= train_df['timestamp'].max(), \
    "데이터 누수 발생!"

print(f"\nTrain: {len(train_df):,}개")
print(f"  기간: {train_df['datetime'].min().date()} ~ {train_df['datetime'].max().date()}")
print(f"\nTest:  {len(test_df):,}개")
print(f"  기간: {test_df['datetime'].min().date()} ~ {test_df['datetime'].max().date()}")
print("\n시간 기준 분리 완료 — 데이터 누수 없음")

중복 제거: 0개 제거됨
정제 후 평점 수: 1,000,209

Train: 800,167개
  기간: 2000-04-25 ~ 2000-12-02

Test:  200,042개
  기간: 2000-12-02 ~ 2003-02-28

시간 기준 분리 완료 — 데이터 누수 없음


In [10]:
# userId, movieId → 0부터 시작하는 인덱스로 변환
user_ids  = train_df['userId'].unique()
movie_ids = train_df['movieId'].unique()

user2idx  = {u: i for i, u in enumerate(user_ids)}
movie2idx = {m: i for i, m in enumerate(movie_ids)}

train_df['user_idx']  = train_df['userId'].map(user2idx)
train_df['movie_idx'] = train_df['movieId'].map(movie2idx)

test_df['user_idx']  = test_df['userId'].map(user2idx).fillna(0).astype(int)
test_df['movie_idx'] = test_df['movieId'].map(movie2idx).fillna(0).astype(int)

n_users  = len(user2idx)
n_movies = len(movie2idx)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"유저 수:  {n_users:,}")
print(f"영화 수:  {n_movies:,}")
print(f"사용 장치: {device}")

# 시간 감쇠 가중치
def compute_temporal_weights(df, lam=0.1):
    t_max  = df['timestamp'].max()
    t_min  = df['timestamp'].min()
    t_half = (t_max - t_min) / 2
    weights = np.exp(-lam * (t_max - df['timestamp']) / (t_half + 1))
    return weights.values

train_weights = compute_temporal_weights(train_df, lam=0.1)
print(f"\n가중치 범위: {train_weights.min():.4f} ~ {train_weights.max():.4f}")
print(f"평균 가중치: {train_weights.mean():.4f}")

# Dataset 클래스
class RatingDataset(Dataset):
    def __init__(self, df, weights=None):
        self.users   = torch.LongTensor(df['user_idx'].values)
        self.movies  = torch.LongTensor(df['movie_idx'].values)
        self.ratings = torch.FloatTensor(df['rating'].values)
        self.weights = torch.FloatTensor(
            weights if weights is not None else np.ones(len(df))
        )

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (self.users[idx], self.movies[idx],
                self.ratings[idx], self.weights[idx])

train_dataset = RatingDataset(train_df, train_weights)
test_dataset  = RatingDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=512, shuffle=False)

print(f"\nTrain 배치 수: {len(train_loader)}")
print(f"Test  배치 수: {len(test_loader)}")
print("Dataset 준비 완료")

유저 수:  5,400
영화 수:  3,662
사용 장치: cpu

가중치 범위: 0.8187 ~ 1.0000
평균 가중치: 0.9273

Train 배치 수: 1563
Test  배치 수: 391
Dataset 준비 완료


In [11]:
class TemporalNCF(nn.Module):
    def __init__(self, n_users, n_movies, embed_dim=32, layers=[64, 32, 16]):
        super().__init__()
        self.user_embedding  = nn.Embedding(n_users,  embed_dim)
        self.movie_embedding = nn.Embedding(n_movies, embed_dim)

        mlp_layers = []
        input_dim = embed_dim * 2
        for out_dim in layers:
            mlp_layers.append(nn.Linear(input_dim, out_dim))
            mlp_layers.append(nn.ReLU())
            mlp_layers.append(nn.Dropout(0.2))
            input_dim = out_dim
        mlp_layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*mlp_layers)

    def forward(self, user_idx, movie_idx, weights=None):
        u = self.user_embedding(user_idx)
        m = self.movie_embedding(movie_idx)
        if weights is not None:
            w = weights.unsqueeze(1)
            u = u * w
            m = m * w
        x = torch.cat([u, m], dim=1)
        out = self.mlp(x)
        out = torch.sigmoid(out) * 4 + 1
        return out.squeeze()

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    criterion = nn.MSELoss(reduction='none')
    for users, movies, ratings, weights in loader:
        users, movies = users.to(device), movies.to(device)
        ratings, weights = ratings.to(device), weights.to(device)
        optimizer.zero_grad()
        preds = model(users, movies, weights)
        loss = (criterion(preds, ratings) * weights).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    preds_list, targets_list = [], []
    with torch.no_grad():
        for users, movies, ratings, weights in loader:
            users, movies = users.to(device), movies.to(device)
            preds = model(users, movies)
            preds_list.extend(preds.cpu().numpy())
            targets_list.extend(ratings.numpy())
    preds_arr   = np.array(preds_list)
    targets_arr = np.array(targets_list)
    rmse = np.sqrt(np.mean((preds_arr - targets_arr) ** 2))
    mae  = np.mean(np.abs(preds_arr - targets_arr))
    return rmse, mae

# 학습 실행
model = TemporalNCF(n_users, n_movies, embed_dim=32, layers=[64, 32, 16])
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

print("Temporal NCF 학습 시작 (20 에폭)\n")
best_rmse  = float('inf')
best_epoch = 0

for epoch in range(1, 21):
    train_loss = train_epoch(model, train_loader, optimizer, device)
    rmse, mae  = evaluate(model, test_loader, device)
    scheduler.step()

    if rmse < best_rmse:
        best_rmse  = rmse
        best_epoch = epoch
        torch.save(model.state_dict(), '../results/best_temporal_ncf_1m.pt')

    if epoch % 5 == 0:
        print(f"Epoch {epoch:2d} | Loss: {train_loss:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")

print(f"\n최적 모델: Epoch {best_epoch}  (RMSE: {best_rmse:.4f})")
print(f"\n=== 비교 ===")
print(f"SVD 베이스라인:  1.0300  (100k 기준)")
print(f"TemporalSVD_v2: 1.0294  (100k 기준)")
print(f"TemporalNCF:    {best_rmse:.4f}  (1M 기준)")

Temporal NCF 학습 시작 (20 에폭)

Epoch  5 | Loss: 0.7661 | RMSE: 1.0373 | MAE: 0.8217
Epoch 10 | Loss: 0.7419 | RMSE: 1.1055 | MAE: 0.8653
Epoch 15 | Loss: 0.7158 | RMSE: 1.1776 | MAE: 0.9188
Epoch 20 | Loss: 0.7007 | RMSE: 1.1797 | MAE: 0.9193

최적 모델: Epoch 2  (RMSE: 0.9679)

=== 비교 ===
SVD 베이스라인:  1.0300  (100k 기준)
TemporalSVD_v2: 1.0294  (100k 기준)
TemporalNCF:    0.9679  (1M 기준)


In [12]:
model2 = TemporalNCF(n_users, n_movies, embed_dim=32, layers=[64, 32, 16])
model2 = model2.to(device)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001, weight_decay=1e-5)

print("Early Stopping 적용 재학습\n")

best_rmse2   = float('inf')
best_epoch2  = 0
patience     = 3
no_improve   = 0

for epoch in range(1, 31):
    train_loss = train_epoch(model2, train_loader, optimizer2, device)
    rmse, mae  = evaluate(model2, test_loader, device)

    if rmse < best_rmse2:
        best_rmse2  = rmse
        best_epoch2 = epoch
        no_improve  = 0
        torch.save(model2.state_dict(), '../results/best_temporal_ncf_final.pt')
    else:
        no_improve += 1

    print(f"Epoch {epoch:2d} | Loss: {train_loss:.4f} | RMSE: {rmse:.4f} | {'★ best' if no_improve==0 else ''}")

    if no_improve >= patience:
        print(f"\nEarly Stopping — {patience}회 연속 미개선")
        break

print(f"\n최적 모델: Epoch {best_epoch2}  (RMSE: {best_rmse2:.4f})")
print(f"\n=== 최종 비교 ===")
print(f"SVD 베이스라인:  1.0300")
print(f"TemporalSVD_v2: 1.0294")
print(f"TemporalNCF:    {best_rmse2:.4f}  ← 딥러닝 최종")

Early Stopping 적용 재학습

Epoch  1 | Loss: 1.0097 | RMSE: 0.9784 | ★ best
Epoch  2 | Loss: 0.8185 | RMSE: 0.9704 | ★ best
Epoch  3 | Loss: 0.7870 | RMSE: 0.9904 | 
Epoch  4 | Loss: 0.7772 | RMSE: 1.0000 | 
Epoch  5 | Loss: 0.7687 | RMSE: 1.0282 | 

Early Stopping — 3회 연속 미개선

최적 모델: Epoch 2  (RMSE: 0.9704)

=== 최종 비교 ===
SVD 베이스라인:  1.0300
TemporalSVD_v2: 1.0294
TemporalNCF:    0.9704  ← 딥러닝 최종
